# Momants intent classification

This notebook trains and uses a separate SetFit classifier for six fixed visitor intents. Training uses only the handwritten examples in `intent_training.csv`.

## 1. Import the intent module

In [ ]:
from pathlib import Path
import importlib
import sys

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import momants_intentie
importlib.reload(momants_intentie)

## 2. Set the paths

In [ ]:
TRAINING_DATA_PATH = PROJECT_DIR / "intent_training.csv"
MODEL_PATH = PROJECT_DIR / "model" / "momants-intentie"
CSV_PATH = PROJECT_DIR / "attached_assets" / "test_gesprekken_100.csv"
OUTPUT_DIRECTORY = PROJECT_DIR / "results"

## 3. Train the model once

Run this cell once. For later classifications, you can skip this step while the `model/momants-intentie` directory exists.

In [ ]:
saved_model = momants_intentie.train_model(
    training_data_path=TRAINING_DATA_PATH,
    model_path=MODEL_PATH,
)
print(f"Model saved to: {saved_model}")

## 4. Validate the Momants export without a model

This step reads only the permitted fields and displays counts only.

In [ ]:
data = momants_intentie.load_momants_csv(CSV_PATH)
visitors = momants_intentie.select_visitor_messages(data)
print(f"Message rows: {len(data)}")
print(f"Usable visitor messages: {len(visitors)}")
print(f"Conversations: {visitors['conversation_id'].nunique()}")

## 5. Classify the intents

This step uses the previously trained local model and writes a file named with the start time, for example `resultaten/intents_per_conversation_20260901_143522_123456.csv`.

In [ ]:
intents = momants_intentie.process_csv(
    csv_path=CSV_PATH,
    output_directory=OUTPUT_DIRECTORY,
    model_path=MODEL_PATH,
    batch_size=32,
)
print(f"Complete: {len(intents)} conversation intents found.")
print(f"File: {intents.attrs['output_path']}")
intents.head(10)